## **PAIR ASSIGNMENT**

Please work in pairs for this exercise. Pair up with the person next to you. If you find that there isn't anyone sitting next to you or if you're unable to form a pair, please raise your hand, and I will assist in pairing you with someone.


### **Task 0: Please assign your full name and your partner's full name to the variables below, respectively.**


Example:

```
your_name = 'Taylor Swift'
your_partner_name = 'Travis Kelce'
```

In [ ]:
# Assign your and your partner's names here.  If you find that there isn't anyone sitting next to you or if you're unable to form a pair, please raise your hand.
# If, in the end, we are unable to find a partner for you, please assign the word "self" to the `your_partner_name` variable.
your_name = ''
your_partner_name = ''

# **Principal Component Analysis (PCA) and Dimensionality Reduction**

In this notebook exercise, we will explore **Principal Component Analysis (PCA)**, a powerful dimensionality reduction technique widely used in machine learning and data science. PCA helps us reduce the number of features in a dataset while retaining most of the important information.

## **Why Dimensionality Reduction?**

High-dimensional datasets can suffer from several problems:
- **Curse of dimensionality**: As dimensions increase, data becomes sparse and distances between points become less meaningful
- **Computational cost**: More features mean longer training times and higher memory requirements
- **Overfitting**: Too many features relative to samples can lead to overfitting
- **Visualization**: Hard to visualize data in more than 3 dimensions

## **What is PCA?**

PCA is an unsupervised learning technique that:
1. Finds the directions (principal components) of maximum variance in the data
2. Projects the data onto these directions
3. Allows us to keep only the top k components, reducing dimensionality from n to k features

The key insight is that many features in high-dimensional data are correlated and redundant. PCA identifies the most important patterns and discards the noise.

## **Objectives**

In this exercise, you will:
1. Load and explore a high-dimensional dataset (MNIST with 784 features)
2. Apply PCA to reduce its dimensionality
3. Visualize the data in reduced dimensions
4. Train classifiers on both original and reduced data
5. Compare their performance

## **Import Libraries**

Let's start by importing the necessary libraries for our analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time

%matplotlib inline
sns.set_style('whitegrid')
# Set matplotlib to always start y-axis at 0 for bar charts and appropriate visualizations
plt.rcParams['axes.autolimit_mode'] = 'round_numbers'
plt.rcParams['axes.xmargin'] = 0
plt.rcParams['axes.ymargin'] = 0

## **Load and Explore the Dataset**

We'll use the MNIST dataset, which contains 28x28 pixel images of handwritten digits (0-9), resulting in 784 features per sample. This is a realistic high-dimensional dataset where PCA can demonstrate significant computational benefits.

**Note**: The first time you run this, it will download the MNIST dataset (~17MB), which may take a moment.

- Load the MNIST dataset
- Store the features in `X` and the target labels in `y`
- Use a subset for faster execution (10,000 samples)
- Print the shape of X and y
- Print the number of features

In [ ]:
print("Loading MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X = mnist.data.to_numpy() if hasattr(mnist.data, 'to_numpy') else mnist.data
y = mnist.target.to_numpy() if hasattr(mnist.target, 'to_numpy') else mnist.target
y = y.astype(int)

print(f"Original Dataset shape: {X.shape}")
# Use a subset for faster execution
np.random.seed(42)
indices = np.random.choice(len(X), size=10000, replace=False)
X = X[indices]
y = y[indices]

print(f"\nDataset shape: {X.shape}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Target classes: {np.unique(y)}")

## **Visualize Some Sample Images**

Let's visualize a few sample digits to understand what our data looks like.

- Use matplotlib to display the first 10 images from the dataset
- Show them in a 2x5 grid

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    # Reshape flat array back to 28x28 image
    image = X[i].reshape(28, 28)
    ax.imshow(image, cmap='gray')
    ax.set_title(f"Label: {y[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## **Split the Data and Standardize**

Before applying PCA, we need to:
1. Split the data into training and testing sets
2. Standardize the features (PCA is sensitive to feature scales)

- Split the data with 30% for testing, random_state=42
- Create a StandardScaler and fit it on the training data
- Transform both training and testing data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nOriginal feature mean: {X_train[0].mean():.2f}")
print(f"Scaled feature mean: {X_train_scaled[0].mean():.2f}")

## **TASK 1: Apply PCA and Analyze Explained Variance**

Now let's apply PCA to understand how much variance is explained by each principal component.

- Create a PCA object without specifying n_components (to get all components)
- Fit it on the scaled training data
- Print the explained variance ratio for the first 10 components by using the `explained_variance_ratio_` member of the PCA object.
- Use the np.cumsum() method to compute the cumulative summation of variance

Create two plots:
1. Individual explained variance for each component
2. Cumulative explained variance

This will help us decide how many components to keep.

In [ ]:
# Write your code here
pca_full = ...

# Cumulative explained variance
cumsum = ...

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Individual explained variance
ax1.plot(range(1, len(pca_full.explained_variance_ratio_) + 1), 
        pca_full.explained_variance_ratio_, 
        color='red')
ax1.set_xlabel('Principal Component', fontsize=12)
ax1.set_ylabel('Explained Variance Ratio', fontsize=12)
ax1.set_title('Individual Explained Variance by Component', fontsize=14)

ax2.plot(range(1, len(cumsum) + 1), cumsum, marker='o', linestyle='-', color='coral', linewidth=2)
ax2.set_xlabel('Number of Components', fontsize=12)
ax2.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax2.set_title('Cumulative Explained Variance', fontsize=14)
ax2.legend()

plt.tight_layout()
plt.show()


## **TASK 2: Apply PCA with Different Numbers of Components**

Let's apply PCA with different numbers of components:
- 2 components (create PCA with n_components=2)
- Number of components for 95% variance (create PCA with n_components=0.95)

Note that the PCA() constructor behaves differently depending on whether it is given a value between `[0,1)` versus an integer greater than one.

Transform both training and testing data for each case.

In [ ]:
# Write your code here

# PCA with 2 components for visualization
pca_2d = ...

# PCA with components for 95% variance
pca_95 = ...


## **TASK 3: Visualize 2D PCA Projection**

Let's visualize our high-dimensional data in 2D using the first two principal components.

- Create a scatter plot of the 2D PCA projection
- Color each point by its digit label
- Add a legend and appropriate labels

Notice that there is a lot of overlap, suggesting that 2 dimensions is likely not enough to represent this input.

In [ ]:
# Write your code here


## **Train Classifiers on Original Data**

Now let's train classifiers on the original (scaled) data to establish a baseline.

- Train a Logistic Regression classifier
- Train a Random Forest classifier
- Measure training time and accuracy for both
- Store results for comparison

In [ ]:
def logistic_regression(X, Y, X_test, Y_test, features_name, model_name):
    start_time = time.time()
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X, Y)
    print(f"Number of iterations to train: {lr.n_iter_}")
    lr_time = time.time() - start_time

    lr_train_acc = accuracy_score(Y, lr.predict(X))
    lr_test_acc = accuracy_score(Y_test, lr.predict(X_test))

    print(f"Training {model_name} with {features_name} features")
    print(f"Train Accuracy: {lr_train_acc:.4f}")
    print(f"Test Accuracy: {lr_test_acc:.4f}")
    print(f"Training Time: {lr_time:.4f}s")

    return {
        'Model': model_name,
        'Features': features_name,
        'Train Time (s)': lr_time,
        'Train Accuracy': lr_train_acc,
        'Test Accuracy': lr_test_acc
    }

def random_forest(X, Y, X_test, Y_test, features_name, model_name):
    start_time = time.time()
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X, Y)
    rf_time = time.time() - start_time

    rf_train_acc = accuracy_score(Y, rf.predict(X))
    rf_test_acc = accuracy_score(Y_test, rf.predict(X_test))

    print(f"Training {model_name} with {features_name} features")
    print(f"Train Accuracy: {rf_train_acc:.4f}")
    print(f"Test Accuracy: {rf_test_acc:.4f}")
    print(f"Training Time: {rf_time:.4f}s")

    return {
        'Model': model_name,
        'Features': features_name,
        'Train Time (s)': rf_time,
        'Train Accuracy': rf_train_acc,
        'Test Accuracy': rf_test_acc
    }

In [ ]:
# Write your code here

# Logistic Regression on original data
print("Training Logistic Regression on original data...")
lr_original_result = ...

# Random Forest on original data
print("\nTraining Random Forest on original data...")
rf_original_result = ...

## **TASK 4: Train Classifiers on PCA-Reduced Data (95% Variance)**

Now let's train the same classifiers on the PCA-reduced data (95% variance).

- Train a Logistic Regression classifier
- Train a Random Forest classifier
- Measure training time and accuracy for both, by using the time.time() method in python to get the current time before and after, and then taking the difference.
- Store results for comparison

In [ ]:
# Write your code here

# Logistic Regression on PCA data (95% variance)
print(f"Training Logistic Regression on PCA data ({pca_95.n_components_} components)...")
lr_pca_result = ...

# Random Forest on PCA data (95% variance)
print(f"\nTraining Random Forest on PCA data ({pca_95.n_components_} components)...")
rf_pca_result = ...

## **TASK 7: Visualize Performance Comparison**

Create visualizations comparing:
1. Test accuracy across models and feature sets
2. Training time across models and feature sets

In [ ]:
# Combine all result objects into a list
all_results = [
    lr_original_result,
    rf_original_result,
    lr_pca_result,
    rf_pca_result
]

# Create DataFrame from the list of result objects
results_df = pd.DataFrame(all_results)

# Write your code here


### **QUESTION**

Why did it take longer to train the PCA-reduced model, even though it has fewer features?

## **TASK 8: Experiment with Different Numbers of Components**

Let's see how performance changes as we vary the number of PCA components.

- Try different numbers of components: [10, 25, 50, 100, 150, 200, 300, 400, 784]
- Train Logistic Regression for each
- Plot accuracy vs number of components

In [ ]:
n_components_list = [10, 25, 50, 100, 150, 200, 300, 400, 784]
train_accuracies = []
test_accuracies = []
train_times = []
explained_variances = []

for n_comp in n_components_list:
    print(f"Testing with {n_comp} components...")

    # Write your code here
    
    # Apply PCA
    ...
    
    # Train model
    ...
    
    print(f"  Explained variance: {explained_var:.4f}")
    print(f"  Test accuracy: {test_acc:.4f}")
    print(f"  Training time: {train_time:.4f}s\n")

## **Visualize the Trade-offs**

Create plots showing:
1. Accuracy vs number of components
2. Training time vs number of components
3. Explained variance vs number of components

In [ ]:
# Write your code here
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Accuracy vs components
axes[0, 0].plot(n_components_list, train_accuracies, marker='o', label='Train Accuracy', linewidth=2)
axes[0, 0].plot(n_components_list, test_accuracies, marker='s', label='Test Accuracy', linewidth=2)
axes[0, 0].set_xlabel('Number of Components', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].set_title('Accuracy vs Number of PCA Components', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0.90, 1.01])

# Training time vs components
axes[0, 1].plot(n_components_list, train_times, marker='o', color='coral', linewidth=2)
axes[0, 1].set_xlabel('Number of Components', fontsize=12)
axes[0, 1].set_ylabel('Training Time (seconds)', fontsize=12)
axes[0, 1].set_title('Training Time vs Number of PCA Components', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Explained variance vs components
axes[1, 0].plot(n_components_list, explained_variances, marker='o', color='green', linewidth=2)
axes[1, 0].axhline(y=0.95, color='r', linestyle='--', label='95% Variance', linewidth=2)
axes[1, 0].set_xlabel('Number of Components', fontsize=12)
axes[1, 0].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[1, 0].set_title('Explained Variance vs Number of PCA Components', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

# Combined view: Accuracy and Time
ax3 = axes[1, 1]
ax3_twin = ax3.twinx()

line1 = ax3.plot(n_components_list, test_accuracies, marker='s', color='steelblue', 
                 label='Test Accuracy', linewidth=2)
line2 = ax3_twin.plot(n_components_list, train_times, marker='o', color='coral', 
                      label='Training Time', linewidth=2)

ax3.set_xlabel('Number of Components', fontsize=12)
ax3.set_ylabel('Test Accuracy', color='steelblue', fontsize=12)
ax3_twin.set_ylabel('Training Time (seconds)', color='coral', fontsize=12)
ax3.set_title('Trade-off: Accuracy vs Training Time', fontsize=14, fontweight='bold')
ax3.tick_params(axis='y', labelcolor='steelblue')
ax3_twin.tick_params(axis='y', labelcolor='coral')

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax3.legend(lines, labels, loc='center right', fontsize=11)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## **Summary and Key Takeaways**

### What we learned:

1. **Dimensionality Reduction with PCA**:
   - PCA can significantly reduce the number of features while retaining most of the variance
   - We reduced 784 features to ~150-200 features while keeping 95% of the variance
   - This represents a ~75-80% reduction in dimensionality

2. **Performance Trade-offs**:
   - **Accuracy**: PCA-reduced data often maintains comparable accuracy to original data (typically 1-3% difference)
   - **Training Time**: Models train **significantly faster** with fewer features (often 2-5x speedup with 95% variance)
   - **Memory Usage**: PCA-reduced data requires less memory for storage and processing
   - **Interpretability**: PCA components are harder to interpret than original pixel features

3. **Why MNIST Shows Better Results**:
   - **High dimensionality**: 784 features vs 64 in digits dataset
   - **More samples**: 10,000 samples makes training time differences more noticeable
   - **Redundancy**: Many pixels in MNIST are highly correlated (edges, background)
   - **Real-world scale**: More representative of practical applications

4. **When to Use PCA**:
   - High-dimensional datasets with redundant features (images, sensors, genomics)
   - When computational efficiency is important (large-scale training)
   - For data visualization (reducing to 2-3 dimensions)
   - When features are correlated
   - To reduce overfitting by eliminating noise

5. **When NOT to Use PCA**:
   - When interpretability of original features is crucial (medical diagnosis)
   - When features are already low-dimensional (<50-100 features)
   - When features are already uncorrelated
   - For sparse data (PCA creates dense representations)
   - When linear relationships don't capture the data structure well

6. **Best Practices**:
   - Always standardize features before applying PCA (mean=0, std=1)
   - Choose number of components based on explained variance ratio (typically 90-95%)
   - Fit PCA on training data only, then transform test data (avoid data leakage)
   - Compare performance with and without PCA
   - Consider the trade-off between speed and accuracy

### Discussion Questions:

1. How did PCA affect the accuracy of your models? Was the trade-off acceptable?
2. What was the impact on training time? How much speedup did you observe?
3. How many components would you choose for a production system? Why?
4. Can you think of other high-dimensional datasets where PCA would be useful?
5. What are the limitations of PCA that we observed?
6. Would non-linear dimensionality reduction (like t-SNE, UMAP) be better for some tasks?